# Netflix Movies and TV Shows — Exploratory Data Analysis

This project explores Netflix movies and TV shows to identify
patterns, distributions, relationships, outliers, and business insights.

In [ ]:
import pandas as pd
import os
import numpy as np
import plotly.express as px
print('Libraries imported successfully!')

Libraries imported successfully!


1. Load Dataset

In [ ]:
file= pd.read_csv('netflix_titles.csv')
print('File imported successfully')
print(f'{file.head(5)}')
os.makedirs("../outputs",exist_ok=True)

File imported successfully
  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN   
3                                                NaN            NaN   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

           date_added  release_year rating   duration  \
0  September 25, 2021          2020  PG-13     90 min   
1  September 24, 2021          2021  TV-MA  2 Seasons   
2 

2. Dataset overview

In [80]:
print(f'Number of rows : {file.shape[0]}')
print(f'Number of columns : {file.shape[1]}')
print(file.dtypes)
print(file.info())
print(file.describe(include='all'))


Number of rows : 8807
Number of columns : 12
show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes:

## 3. Data Quality Analysis

Before performing deeper analysis, we check:

- Missing values
- Duplicate records
- Data types
- Unique values
- Invalid or unusual values

In [81]:
missing_values= file.isna().sum()
missing_values

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [82]:
missing_percentage=(
    file.isna().mean().mul(100).sort_values(ascending=False)
)
missing_percentage

director        29.908028
country          9.435676
cast             9.367549
date_added       0.113546
rating           0.045418
duration         0.034064
show_id          0.000000
type             0.000000
title            0.000000
release_year     0.000000
listed_in        0.000000
description      0.000000
dtype: float64

In [83]:
missing_summary= pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})
missing_summary= missing_summary.sort_values(
    "Missing Percentage",
    ascending=False
)
missing_summary

,Missing Values,Missing Percentage
director,2634,29.908028
country,831,9.435676
cast,825,9.367549
date_added,10,0.113546
rating,4,0.045418
duration,3,0.034064
description,0,0.000000
listed_in,0,0.000000
release_year,0,0.000000
show_id,0,0.000000


In [ ]:
missing_plot= missing_percentage[
    missing_percentage > 0
].reset_index()
missing_plot.columns = ["Column", "Missing Percentage (%)"]
fig= px.bar(missing_plot, title="Missing Values by Column", x='Missing Percentage (%)', y="Column")
fig.show()
fig.write_image("../outputs/missing_values.png")


In [85]:
duplicates= file.duplicated().sum()
print(f"Number of duplicated rows: {duplicates}")
if(duplicates > 0):
    print(file[file.duplicated()])
else:
    print("No duplicated rows found.")

Number of duplicated rows: 0
No duplicated rows found.


## 4. Categorical Data Exploration

In [86]:
print(f'Content Types:{file["type"].unique()}')
print(file["type"].value_counts())

content_type_percentage=(
    file["type"].value_counts(normalize=True).mul(100).round(2)
)
content_type_percentage

Content Types:<ArrowStringArray>
['Movie', 'TV Show']
Length: 2, dtype: str
type
Movie      6131
TV Show    2676
Name: count, dtype: int64


type
Movie      69.62
TV Show    30.38
Name: proportion, dtype: float64

## 5. Movies vs TV Shows

In [ ]:
content_type= file["type"].value_counts().reset_index()
content_type.columns = ["Content Type", "Number of titles"]
fig= px.bar(content_type, title="Content Type Distribution", x="Content Type", y="Number of titles")
fig.show()
fig.write_image("../outputs/content_type_bar.png")

fig = px.pie(
    content_type,
    names="Content Type",
    values="Number of titles",
    title="Content Type Distribution"
)
fig.show()
fig.write_image("../outputs/content_type_pie.png")


## 6. Genre Analysis

In [ ]:
genre_df = file.assign(
    genre=file["listed_in"].str.split(", ")
).explode("genre").reset_index(drop=True)

# Remove rows where genre or type is missing
genre_df = genre_df.dropna(subset=["genre", "type"])

# Count genres
genre_count = genre_df["genre"].value_counts()
top_genre = genre_count.head(10)

fig = px.bar(
    top_genre,
    title="Top 10 Genres",
    x=top_genre.index,
    y=top_genre.values
)

fig.show()
fig.write_image("../outputs/top_ten_genre.png")


# Create genre vs type table
genre_type = pd.crosstab(
    genre_df["genre"],
    genre_df["type"]
)

print(genre_type.head())

# Sort by Movie
genre_type.sort_values(
    by="Movie",
    ascending=False
).head(10)

type                      Movie  TV Show
genre                                   
Action & Adventure          859        0
Anime Features               71        0
Anime Series                  0      176
British TV Shows              0      253
Children & Family Movies    641        0


type,Movie,TV Show
genre,,
International Movies,2752,0
Dramas,2427,0
Comedies,1674,0
Documentaries,869,0
Action & Adventure,859,0
Independent Movies,756,0
Children & Family Movies,641,0
Romantic Movies,616,0
Thrillers,577,0



## 7. Country Analysis

In [ ]:
file["country"].head(10)
country_df= file.assign(
    country=file["country"].str.split(", ")
).explode("country").reset_index(drop=True)
# Remove rows where country is missing
country_df = country_df.dropna(subset=["country"])

country_count= country_df["country"].value_counts()
top_country= country_count.head(10)
fig= px.bar(
    top_country,
    x=top_country.index,
    y=top_country.values,
    title="Top 10 Countries"
)
fig.show()
fig.write_image("../outputs/top_ten_country.png")


## 8. Movie duration Analysis

In [ ]:
movies= file[file["type"]=="Movie"]
movies[["title", "duration"]].head(10)
movies["duration_minutes"] = (
    movies["duration"]
    .str.extract(r"(\d+)")
    .astype(float)
)

movies[[
    "title",
    "duration",
    "duration_minutes"
]].head(10)
duration_stats= movies["duration_minutes"].describe()
print("Mean:", movies["duration_minutes"].mean())
print("Median:", movies["duration_minutes"].median())
print("Minimum:", movies["duration_minutes"].min())
print("Maximum:", movies["duration_minutes"].max())
print("Standard Deviation:", movies["duration_minutes"].std())
print("Skewness:", movies["duration_minutes"].skew())
fig= px.histogram(
    movies,
    x="duration",
    nbins=13,
    title="Movie Duration Analysis"
)
fig.show()
fig.write_image("../outputs/movie_duration_analysis.png")



Mean: 99.57718668407311
Median: 98.0
Minimum: 3.0
Maximum: 312.0
Standard Deviation: 28.290593447417397
Skewness: 0.20257911230639258


## 9. Movie Duration Distribution

In [ ]:
fig= px.histogram(
    movies["duration_minutes"].dropna(),
    nbins=30,
    title="Distribution of Movie Durations (in minutes)",
    labels={
        "duration_minutes": "Duration (minutes)",
        "count": "Number of Movies"
    }
)
fig.show()
fig.write_image("../outputs/movie_duration_distribution.png")


## 10. Outlier Analysis

In [92]:
duration_data= movies["duration_minutes"].dropna()

Q1= duration_data.quantile(0.25)
Q3= duration_data.quantile(0.75)

IQR= Q3-Q1

lower_bound= Q1- 1.5 * IQR
upper_bound= Q3 + 1.5 * IQR

print(Q1, Q3, IQR, lower_bound, upper_bound)

outliers= movies[(movies["duration_minutes"]<lower_bound) | 
                 (movies["duration_minutes"]>upper_bound)].copy()
print(f"Number of outliers {len(outliers)}")
outliers[["title", "duration", "duration_minutes"]].sort_values(
    "duration_minutes", ascending=False
).head(10)

87.0 114.0 27.0 46.5 154.5
Number of outliers 450


,title,duration,duration_minutes
4253,Black Mirror: Bandersnatch,312 min,312.0
717,Headspace: Unwind Your Mind,273 min,273.0
2491,The School of Mischief,253 min,253.0
2487,No Longer kids,237 min,237.0
2484,Lock Your Girls In,233 min,233.0
2488,Raya and Sakina,230 min,230.0
166,Once Upon a Time in America,229 min,229.0
7932,Sangam,228 min,228.0
1019,Lagaan,224 min,224.0
4573,Jodhaa Akbar,214 min,214.0


## 11. Rating Analysis

In [ ]:
rating_counts= file["rating"].value_counts().reset_index()
rating_counts.columns=["rating", "count"]
print(rating_counts)
fig=px.bar(rating_counts,
           x="rating",
           y="count",
           orientation="h",
           title="Distribution of Ratings",
           labels={
               "rating": "Rating",
               "count": "Number of Titles"
           },
           color="count")
fig.show()
fig.write_image("../outputs/ratings-distribution.png")


      rating  count
0      TV-MA   3207
1      TV-14   2160
2      TV-PG    863
3          R    799
4      PG-13    490
5      TV-Y7    334
6       TV-Y    307
7         PG    287
8       TV-G    220
9         NR     80
10         G     41
11  TV-Y7-FV      6
12     NC-17      3
13        UR      3
14    74 min      1
15    84 min      1
16    66 min      1


## 12. Rating by Content Type

In [ ]:
rating_type= pd.crosstab(
    file["rating"],
    file["type"]
)
fig= px.bar(
    rating_type,
    x=rating_type.index,
    y=rating_type.columns,
    title="Rating Distribution by Type",
)
fig.show()
fig.write_image("../outputs/rating_dist_by_type.png")

rating_type_percentage= pd.crosstab(
    file["rating"],
    file["type"],
    normalize="index"
) * 100
print(f"Rating Distribution by Type (%): {rating_type_percentage.round(2)}")

Rating Distribution by Type (%): type       Movie  TV Show
rating                   
66 min    100.00     0.00
74 min    100.00     0.00
84 min    100.00     0.00
G         100.00     0.00
NC-17     100.00     0.00
NR         93.75     6.25
PG        100.00     0.00
PG-13     100.00     0.00
R          99.75     0.25
TV-14      66.06    33.94
TV-G       57.27    42.73
TV-MA      64.30    35.70
TV-PG      62.57    37.43
TV-Y       42.67    57.33
TV-Y7      41.62    58.38
TV-Y7-FV   83.33    16.67
UR        100.00     0.00


## 13. Release Year Analysis

In [ ]:
year_counts=(file["release_year"].value_counts().sort_index())
print(year_counts.tail(10))
fig= px.bar(
    year_counts,
    x=year_counts.index,
    y=year_counts.values,
    title="Distribution of Titles by Release Year"
)
fig.show()
fig.write_image("../outputs/title_dist_by_releaseyear.png")

year_type= pd.crosstab(
    file["release_year"],
    file["type"]
)
fig= px.bar(
    year_type,
    x=year_type.index,
    y=year_type.columns,
    title="Movies vs TV shows by Release Year"
)
fig.show()
fig.write_image("../outputs/movies_vs_tvshow_releaseyear.png")


release_year
2012     237
2013     288
2014     352
2015     560
2016     902
2017    1032
2018    1147
2019    1030
2020     953
2021     592
Name: count, dtype: int64


## 14. Content Added to Netflix Over Time

In [ ]:
file["date_added"]= pd.to_datetime(
    file["date_added"],
    errors="coerce"
)
file[["title", "date_added"]].head(10)
print("Missing date_added values: ", file["date_added"].isna().sum())
file["year_added"]= file["date_added"].dt.year
content_added = file["year_added"].value_counts().sort_index()
fig= px.bar(
    content_added,
    x=content_added.index,
    y=content_added.values,
    title="Distribution of Titles by Year Added to Netflix"
)
fig.show()
fig.write_image("../outputs/title_dist_by_year.png")

added_by_type= pd.crosstab(
    file["year_added"],
    file["type"]
)
fig= px.bar(
    added_by_type,
    x=added_by_type.index,
    y=added_by_type.columns,
    title="Movies vs TV Shows Added by Year"
)
fig.show()
fig.write_image("../outputs/movies_vs_tvshow_year.png")


Missing date_added values:  98


## 15. Relationship Analysis

In [ ]:
movie_analysis= movies[
    ["release_year", "duration_minutes"]
].dropna()
movie_analysis.head()
fig= px.scatter(
    movie_analysis,
    x="release_year",
    y="duration_minutes",
    title="Distribution of Movie Durations by Release Year"
)
fig.show()
fig.write_image("../outputs/movie_duration_dist.png")


## 16. Correlation Analysis

In [ ]:
correlation= movie_analysis.corr(numeric_only= True)
fig= px.imshow(
    correlation,
    text_auto=True,
    title="Correlation Matrix"
)
fig.show()
fig.write_image("../outputs/corrln_matrix.png")


## 17. Advanced Analysis — Directors

In [ ]:
director_df= file[file["director"].notna()].assign(
    director= file["director"].str.split(", ")
).explode("director").reset_index(drop=True)
director_count= director_df["director"].value_counts()
top_directors=director_count.head(10)
fig= px.bar(
    top_directors,
    x=top_directors.index,
    y=top_directors.values,
    title="Top 10 Directors by Number of Titles"
)
fig.show()
fig.write_image("../outputs/top_10_directors.png")


## 18. Advanced Analysis — Cast

In [ ]:
cast_df= file[file["cast"].notna()].assign(
    cast= file["cast"].str.split(", ")
).explode("cast").reset_index(drop=True)
cast_counts= cast_df["cast"].value_counts()
top_cast= cast_counts.head(10)
fig= px.bar(
    top_cast,
    x= top_cast.index,
    y= top_cast.values,
    title="Top 10 Cast Members by Number of Titles"
)
fig.show()
fig.write_image("../outputs/top_10_castmembers.png")


## 19. Genre by Content Type

In [ ]:
genre_type_count=pd.crosstab(
    genre_df["genre"], 
    genre_df["type"]
)

top_genre_type= genre_type_count.head()
fig=px.bar(
    top_genre_type,
    x= top_genre_type.index,
    y= top_genre_type.columns,
    title="Top genre by content type"
)
fig.show()
fig.write_image("../outputs/top_genre_contentType.png")
